# Samudra Emulator: Forward Ocean Heat Content Rollout with ACCESS-OM2

This notebook is a structural departure from `AutoEncoder_om2.ipynb`, not an extension
of it. The original notebook trains a single-timestep reconstruction model: one
`(ocean_heat_content_2d, total_surface_heat_flx)` frame in, the same frame out. This
notebook instead trains an **autoregressive forward emulator**, following the framing in
Subel & Zanna (2024), Dheeshjith et al. (2024), and Samudra (Dheeshjith et al. 2025):
given a short window of past OHC states and the current surface heat flux as a forcing
input, predict the next OHC state. Chaining this forward repeatedly, conditioned only on
a single initial condition and a supplied flux trajectory, produces a continuous OHC
rollout.

**What is carried over from `AutoEncoder_om2.ipynb` essentially unchanged:**
- The `UNet` / `PartialConv2d` architecture from `Emulator.py` for land handling.
- The PET normalisation (`build_normalisation`, `Spatial_climatology`) and the
  `ACCESS_OHC` accessor.
- The general `LightningWrapper` / `L.Trainer` training pattern.

**What changes, and why:**
- **Pipeline structure.** The single-frame pipeline is replaced with a PET-native
  `PipelineBranchPoint` that forks the same upstream sample into a *state* branch
  (windowed in time via `petpipe.modifications.TemporalWindow`) and a *forcing* branch
  (surface flux, supplied but never reconstructed). This is the PET-idiomatic
  replacement for a hand-rolled windowing wrapper — see the section below for why
  `TemporalWindow` rather than `DateRange` is the right tool here.
- **Train / validation / test split.** Following Samudra's evaluation protocol, the
  notebook defines *three* regimes rather than two: a training split, a held-out
  **skill-test rollout** against real forcing (to measure trend-tracking skill), and a
  separate **control rollout** against repeated, near-zero-net-flux forcing (to isolate
  pure equilibrium drift from genuine forced trend response). These measure different
  failure modes and must not be conflated into a single long rollout.
- **Loss function.** Single-step masked MSE is replaced with a multi-step recurrent
  rollout loss, following Subel & Zanna (2024, Eq. 1) and Samudra (Eq. 2).
- **Architecture variants.** The base forward UNet is kept as the reference model, with
  three additional inline-defined variants — a PINN-style heat-budget consistency term,
  a lightweight ensemble/diffusion-flavoured stochastic head, and a closed-loop
  drift-nudging term loosely RL-adjacent — wired in as swappable `loss_mode` options
  rather than a forced single choice. These are frontier options without a literature
  track record on this exact problem; the point of this notebook is to characterise
  *where* the deterministic baseline drifts first, so that the choice between them is
  driven by which failure mode is actually observed, not by a priori preference.

This notebook intentionally does not attempt to fix drift by exposing the model to
out-of-distribution forcing data (cf. the transfer-learning approach in Dheeshjith et
al. 2024). The goal is a self-contained statistical predictor with honest uncertainty
bounds, not one corrected post hoc with target-distribution data.


In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: import dependencies and define paths/device for the full
# forward-emulator workflow. Extends the import list from AutoEncoder_om2.ipynb
# with petpipe.modifications and petpipe.branching, used below to build the
# sliding-window / state-forcing-split pipeline.
# Outcome: pipeline/data/model/plot modules are loaded and local utils are importable.
# -----------------------------------------------------------------------------
# System and path handling
import sys
import functools
from pathlib import Path

# Data handling
import numpy as np
import pandas as pd
import xarray as xr

# PyEarthTools pipeline
import pyearthtools.data as petdata
import pyearthtools.pipeline as petpipe
import pyearthtools.training

# Deep learning
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import lightning as L

# Visualization
import matplotlib.pyplot as plt

# Configure device and random seed
torch.manual_seed(42)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Source data and code directories
userbase = "/g/data/nm47/txs156/"
# Taimoor's repobase:
repobase = "/home/156/txs156/uom/OM2-emulator/"
# Ryan's repobase:
# repobase = "/home/561/rmh561/ML/OM2-emulator/"
# Navid's repobase:
# repobase = "/home/552/nc3020/gdata/OM2-emulator/"

# Add the src directory to the Python path (relative to notebook location)
src_path = Path(repobase) / "src"
sys.path.insert(0, str(src_path))

# Import local OM2 emulator modules
# Same building blocks as AutoEncoder_om2.ipynb -- the UNet and PartialConv2d
# architecture are kept as-is. AutoEncoder is no longer used: the forward emulator
# is not a reconstruction model, so there is no single-frame autoencoding step.
from Data import ACCESS_OHC, build_normalisation, make_fast_dl
from Emulator import LightningWrapper, PartialConv2d, UNet
import warnings


## Data & normalisation

Unchanged from `AutoEncoder_om2.ipynb`. The normalisation strategy and the OHC/flux
variable list are independent of whether the model reconstructs one frame or predicts
the next one, so this step does not need to change.

In [ ]:
# Setup the normalisation
time_start = '2000-01'
train_end = "2015-03"
val_start = "2015-04"
time_end = '2018-12'
time_interval = '1MS'
norm_variables = ["ocean_heat_content_2d", "total_surface_heat_flx"]
norm_strat = "Spatial_climatology"
datapath = userbase + "OM2-emulator/data/1deg_ocean_heat_emulator_data.nc"

mask, normalisation = build_normalisation(datapath, norm_strat, norm_variables, time_window = dict(start=time_start, end=time_end, freq=time_interval),\
                                          train_end = train_end, mask = True)
mean = normalisation._initialisation["mean"]
deviation = normalisation._initialisation["deviation"]


In [ ]:
# Set up accessor for the ACCESS data
ACCESS_OHC_accessor = ACCESS_OHC(
    ["area_t", "ocean_heat_content_2d", "total_surface_heat_flx"],
    root=userbase + "OM2-emulator/data/",
)


## Setup the forward-emulation PET pipeline

This is the core structural change from `AutoEncoder_om2.ipynb`. The original pipeline
iterates over single dates and emits one `(OHC, flux)` frame per sample, which is fine
for reconstruction but has no notion of "predict the next state given recent history and
forcing."

A hand-rolled windowing wrapper around `DateRange` was the first instinct here, but PET
already has purpose-built, tested classes for exactly this pattern, in
`pyearthtools.pipeline.modifications` (`petpipe.modifications`):

- **`TemporalWindow`** — built specifically, per its own docstring, "to provide the
  ability to perform sequence-to-sequence modelling from a data accessor or pipeline
  that was designed to produce single time steps." Given `prior_indexes` and
  `posterior_indexes` (each multiplied by a `timedelta` and applied to the queried
  reference date), it returns a `(prior, posterior)` tuple directly. This is the natural
  fit for "two past states in, one future state out," matching the
  `Φ̃_{t+(n-1)Δt}, Φ̃_{t+nΔt} → Φ̃_{t+(n+1)Δt}` recurrence in Samudra (their Eq. 1).
- **`SequenceRetrieval` / `TemporalRetrieval`** — the more general, lower-level
  mechanism `TemporalWindow` is built on. Useful if a non-contiguous or asymmetric
  sampling pattern is ever needed, but `TemporalWindow`'s narrower interface is the
  better fit for this fixed two-in-one-out window and is used below.

For separating OHC (a *state* to be windowed and predicted) from surface flux (a
*forcing* that is supplied, never reconstructed), PET's `petpipe.branching` module
provides `PipelineBranchPoint`: it forks the same upstream sample down independent
sub-pipelines and returns their results as a tuple, in declared order. This replaces
what would otherwise be a manual "predict everything, then mask flux out of the loss"
workaround with two clean, independently-composed branches:

1. a **state branch**: `SelectDataset(["ocean_heat_content_2d"])` &rarr; `TemporalWindow`
2. a **forcing branch**: `SelectDataset(["total_surface_heat_flx"])`, supplying only the
   forcing at the timesteps the rollout actually needs

Each branch is itself an ordinary PET pipeline, so the existing transforms, the
normalisation step, `FillNan`, and `ToNumpy` conversion are reused unchanged within each
branch — nothing about those steps needs to be rewritten, only re-arranged around the
new branch structure.

In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: define the two re-usable sub-pipeline step sequences (state, forcing)
# that will be forked from the same upstream accessor inside a PipelineBranchPoint.
# Outcome: `state_steps` and `forcing_steps` are tuples of PET pipeline steps, each
# independently valid as the body of a Pipeline(...) call.
# -----------------------------------------------------------------------------

# Both branches start from the same coordinate clean-up as AutoEncoder_om2.ipynb,
# then diverge at the variable-selection step.
_shared_coord_drop = petdata.transforms.coordinates.Drop(['geolat_t', 'geolon_t'])
_shared_var_drop = petdata.transforms.variables.Drop(['area_t'])

# State branch: OHC only. This is the field that is windowed in time (two past
# states as input context, matching Samudra's 2-input/2-output recurrence) and is
# the model's prediction target.
state_steps = (
    _shared_coord_drop,
    _shared_var_drop,
    petpipe.operations.xarray.select.SelectDataset(["ocean_heat_content_2d"]),
    petpipe.operations.xarray.Sort(order=["ocean_heat_content_2d"], strict=True),
    petpipe.operations.xarray.reshape.Dimensions(["time", "latitude", "longitude"]),
    normalisation,
    petdata.transforms.coordinates.Drop(['geolat_t', 'geolon_t']),
    petpipe.operations.xarray.values.FillNan(0, posinf=0, neginf=0),
    petpipe.operations.xarray.conversion.ToNumpy(),
    petpipe.operations.numpy.reshape.Rearrange('c t h w -> t c h w'),
)

# Forcing branch: surface heat flux only. This is a known boundary condition the
# emulator conditions on -- per Subel & Zanna (2024), surface forcing (alongside wind
# stress, not present in this 2D OHC-only dataset) is what prevents an emulator from
# drifting in the absence of any external driving signal. It is never reconstructed,
# so it is not windowed across (t-dt, t) the way the state branch is -- only the
# forcing at the *target* timestep is needed to predict that timestep's OHC.
forcing_steps = (
    _shared_coord_drop,
    _shared_var_drop,
    petpipe.operations.xarray.select.SelectDataset(["total_surface_heat_flx"]),
    petpipe.operations.xarray.Sort(order=["total_surface_heat_flx"], strict=True),
    petpipe.operations.xarray.reshape.Dimensions(["time", "latitude", "longitude"]),
    normalisation,
    petdata.transforms.coordinates.Drop(['geolat_t', 'geolon_t']),
    petpipe.operations.xarray.values.FillNan(0, posinf=0, neginf=0),
    petpipe.operations.xarray.conversion.ToNumpy(),
    petpipe.operations.numpy.reshape.Rearrange('c t h w -> t c h w'),
)


In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: define the rollout time step and the TemporalWindow that turns the
# state branch from single-frame retrieval into (prior, posterior) sequence pairs.
# Outcome: `rollout_timedelta` and `state_window` are defined; `state_window` is a
# PET PipelineIndex step that, given a reference date, returns
# ([state(t-dt), state(t)], [state(t+dt)]).
# -----------------------------------------------------------------------------

# Monthly cadence, matching time_interval = '1MS' used for normalisation above.
rollout_timedelta = petdata.time.TimeDelta((1, "month"))

# prior_indexes=[-1, 0]  -> retrieves state at (t-dt, t): two most recent states,
#                           analogous to using model time tendencies in PDE-based
#                           integration (Samudra Sec 2.2).
# posterior_indexes=[1]  -> retrieves state at (t+dt): the single-step prediction
#                           target. Multi-step rollout during training is handled
#                           separately, below, by re-querying the pipeline at
#                           successive reference dates rather than by widening this
#                           window -- see the rollout-loss section.
#
# merge_method concatenates each retrieved group along the time axis (functools.partial
# pattern taken directly from PyEarthTools' own TemporalWindowDemo.ipynb).
state_window = petpipe.modifications.TemporalWindow(
    prior_indexes=[-1, 0],
    posterior_indexes=[1],
    timedelta=rollout_timedelta,
    merge_method=functools.partial(np.concatenate, axis=0),
)


In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: assemble the full forward-emulation pipeline, branching the shared
# upstream accessor into the windowed state branch and the unwindowed forcing
# branch via PipelineBranchPoint.
# Outcome: `pipeline_i[date]` returns a nested tuple
#   ( (prior_states, posterior_states), forcing_at_t )
# where prior_states has shape (2, 1, H, W) [two timesteps, one OHC channel],
# posterior_states has shape (1, 1, H, W), and forcing_at_t has shape (1, 1, H, W).
# Branch order is declared order, per PipelineBranchPoint.apply -- state branch
# first, forcing branch second -- so this nesting is exact, not incidental.
# -----------------------------------------------------------------------------

pipeline_i = petpipe.Pipeline(
    ACCESS_OHC_accessor,
    petpipe.branching.PipelineBranchPoint(
        (*state_steps, state_window),   # branch 0: windowed OHC state
        forcing_steps,                  # branch 1: flux forcing, single timestep
    ),
    iterator=petpipe.iterators.DateRange(time_start, time_end, interval="1 month"),
)


## Define the training, skill-test, and control-rollout splits

Samudra's evaluation protocol uses two separate long rollouts, deliberately kept apart
because they isolate different failure modes:

1. An **8-year skill-test rollout** against real, held-out atmospheric/surface forcing,
   used to measure how well the emulator tracks genuine forced trends (their Sec 2.5,
   "we take our initial conditions from 2014-09-30 and produce an 8-year rollout using
   the corresponding atmospheric forcing").
2. A **century-scale control rollout** against *repeated* forcing from a window chosen
   specifically for near-zero net heat flux, "allowing the emulators to run for
   arbitrarily long periods of time while minimizing potential temperature drift" (their
   Sec 2.5). This isolates pure equilibrium/numerical drift from any drift caused by a
   genuinely changing forcing signal -- exactly the distinction raised earlier as the
   first thing to characterise before choosing between PINN / diffusion / RL-style
   architecture variants for drift correction.

Conflating these into a single long rollout against real forcing would make it
impossible to tell whether divergence from ground truth reflects the model failing to
track a real trend, or the model drifting under flat forcing -- so both are defined
explicitly below, using PET's native `DateRange` and `SuperIterator` (chaining the same
`DateRange` window back-to-back), rather than a hand-rolled repeat loop.

The training split itself is unchanged in spirit from `AutoEncoder_om2.ipynb` -- same
PET `DateRange` iterator class, same general boundary dates -- only the validation
window is trimmed slightly to leave room for the two held-out rollout regimes.

In [ ]:
# We define the training / validation / skill-test / control-rollout splits here.
#
# `train_split`   -- used for gradient updates.
# `valid_split`   -- used for early stopping / monitoring during training, as before.
# `skill_test_split` -- held-out window with REAL forcing, for measuring trend-tracking
#                        skill via an autoregressive rollout (Samudra Sec 2.5, "8-year
#                        rollout").
# `control_window`   -- a single near-zero-net-flux window, chosen the same way as
#                        Samudra's control run, to be REPEATED via SuperIterator below
#                        rather than iterated once.

splits = {
    "train_split": petpipe.iterators.DateRange(
        "2000-01", "2014-01", interval="1 month"
    ),
    "valid_split": petpipe.iterators.DateRange(
        "2014-01", "2015-04", interval="1 month"
    ),
}

# Held-out skill-test window: real forcing, used for an autoregressive rollout
# (see the "Skill-test rollout" section near the end of this notebook). Kept
# separate from `splits` since it is not used for training-time gradient updates
# or validation-loss monitoring -- it is consumed directly by the rollout loop.
skill_test_split = petpipe.iterators.DateRange(
    "2015-04", "2019-01", interval="1 month"
)

# Control-rollout window: a single decade chosen for near-zero net surface heat
# flux, the same rationale as Samudra Sec 2.5. The actual near-zero-flux window
# for this dataset should be identified empirically from `total_surface_heat_flx`
# (e.g. by inspecting its area-weighted annual mean over the training period) --
# the date range below is a placeholder to be confirmed against the real data
# before the control rollout is run.
control_window = petpipe.iterators.DateRange(
    "2005-01", "2015-01", interval="1 month"
)

# Repeat the control window N times via SuperIterator, chaining the same
# DateRange back-to-back -- this is the PET-native equivalent of Samudra's
# "repeat 10-year cycle" control protocol, built from the existing DateRange
# class rather than a bespoke repeating-iterator wrapper.
n_control_repeats = 10  # 10 repeats of a 10-year window = a century-scale rollout,
                         # matching the order of magnitude of Samudra's century run.
control_iterator = petpipe.iterators.SuperIterator(
    *([control_window] * n_control_repeats)
)


## Define the forward-emulator architecture

The base `UNet` and `PartialConv2d` land-handling from `Emulator.py` are kept as-is --
the relevant change is to the channel wiring at the model's input and output heads, not
to the encoder/decoder blocks themselves. The original reconstruction model used
`UNet(input_channel_count=len(norm_variables), output_channel_count=len(norm_variables))`,
i.e. 2 channels in, 2 channels out (OHC and flux, both reconstructed).

The forward emulator's channel contract follows Samudra's 2-input/2-output recurrence
(their Eq. 1), adapted to a single 2D state variable: two past OHC states
(`prior_indexes=[-1, 0]` from the pipeline above) plus the current surface flux forcing
go in; one future OHC state comes out.

- **Input channels: 3** -- `OHC(t-dt)`, `OHC(t)`, `flux(t)`.
- **Output channels: 1** -- `OHC(t+dt)`.

`ForwardUNet` below is a thin wrapper around the existing `UNet`, doing only the
channel-count rewiring and the tensor reshaping needed to go from the pipeline's nested
`(prior_states, posterior_states), forcing` structure to a single stacked-channel input
tensor. It is not a new architecture.

In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: thin wrapper around the existing UNet, rewiring channel counts for
# the forward-emulation contract (2 past states + forcing in, 1 future state out)
# and handling the reshape from the pipeline's nested output structure.
# -----------------------------------------------------------------------------

class ForwardUNet(nn.Module):
    """
    Forward one-step OHC emulator.

    Wraps the existing UNet from Emulator.py unchanged at the block level; only
    the input/output channel counts differ from the reconstruction model in
    AutoEncoder_om2.ipynb. PartialConv2d land-handling is inherited from the base
    UNet implementation and is not modified here.

    forward() expects:
        prior_states : (B, 2, H, W)  -- OHC(t-dt), OHC(t), channel-stacked
        forcing      : (B, 1, H, W)  -- flux(t)
    and returns:
        next_state   : (B, 1, H, W)  -- predicted OHC(t+dt)
    """

    def __init__(self, n_prior_states: int = 2, n_forcing_channels: int = 1):
        super().__init__()
        self.n_prior_states = n_prior_states
        self.n_forcing_channels = n_forcing_channels
        self.base_unet = UNet(
            input_channel_count=n_prior_states + n_forcing_channels,
            output_channel_count=1,
        )

    def forward(self, prior_states: torch.Tensor, forcing: torch.Tensor) -> torch.Tensor:
        x = torch.cat([prior_states, forcing], dim=1)
        return self.base_unet(x)

### Frontier architecture variants

The deterministic `ForwardUNet` above is the reference model used to characterise
out-of-sample drift first, per the plan agreed above. The three variants below are
defined inline as swappable options on top of that same base model, selected via the
`loss_mode` argument to the Lightning wrapper later in this notebook -- not as a forced
single choice, since which one is best-matched depends on *which* failure mode the
baseline's drift diagnostics actually show:

- **PINN-style heat-budget consistency term.** Best matched to *trend-magnitude
  attenuation* (a systematically too-damped response under real forcing, the kind Samudra
  reports: "for most depths the trained models underestimate trends by 20% to 50%
  relative to OM4"). A standard MSE rollout loss biases toward the conditional mean,
  which under-represents a small forced trend riding on much larger internal
  variability. The term below penalises the discrepancy between the model's actual OHC
  tendency and a flux-integrated heat-budget estimate of that tendency, giving the model
  a direct incentive to match cumulative heat input rather than only minimising per-step
  squared error.
- **Ensemble / diffusion-flavoured stochastic head.** Best matched to *equilibrium
  variance drift* under flat/repeated forcing -- compounding error in a chaotic
  autoregressive system, which a single deterministic trajectory cannot represent
  honestly. Rather than a full diffusion sampler, this is implemented as a lightweight
  noise-conditioned stochastic head: the same base UNet, several independently-seeded
  stochastic forward passes per step, read as an ensemble. This keeps the predictor
  self-contained (uncertainty is learned from the model's own training distribution, not
  imported from out-of-sample data) while being cheap enough to run alongside the other
  variants for comparison.
- **Closed-loop drift-nudging term, RL-adjacent.** A full RL formulation needs a reward
  signal, and the natural reward for "stay close to ground truth" is just the supervised
  rollout loss with extra steps -- so this is implemented as a softer closed-loop bias
  correction (nudging the rollout's long-run mean toward a climatological constraint, in
  the spirit of Samudra's `Q_anom` channel separating climatological-mean forcing from
  anomaly) rather than a full RL controller. This is the variant to revisit properly,
  with an actual reward/controller formulation, only if the cheaper levers above turn out
  not to address the observed drift -- not the first thing to reach for.

In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: PINN-style heat-budget consistency term.
# Penalises the discrepancy between the model's predicted OHC tendency and a
# flux-integrated estimate of that tendency, as a soft constraint added to the
# base rollout loss (see rollout_loss / loss_mode="pinn" below).
# -----------------------------------------------------------------------------

def heat_budget_residual(
    ohc_t: torch.Tensor,
    ohc_t_plus_dt: torch.Tensor,
    flux_t: torch.Tensor,
    dt_seconds: float,
    mask: torch.Tensor,
) -> torch.Tensor:
    """
    Heat-budget consistency residual: d(OHC)/dt implied by the model's own
    prediction, vs. the flux-integrated estimate of that tendency.

    This is a soft constraint, not a strict conservation law -- a 2D depth-
    integrated OHC field cannot resolve lateral heat transport, so the residual
    is expected to be non-zero in general (consistent with Samudra's own note
    that "strict conservation is not imposed" in their full-depth model, Sec 3.1).
    The point is to penalise large, systematic deviations -- the kind of bias
    that produces trend attenuation -- not to drive the residual exactly to zero.

    Args:
        ohc_t, ohc_t_plus_dt : (B, 1, H, W), de-normalised physical units (J/m^2)
        flux_t               : (B, 1, H, W), de-normalised physical units (W/m^2)
        dt_seconds            : timestep length in seconds (e.g. ~2.628e6 for 1 month)
        mask                  : (H, W) ocean mask, 1 = ocean, 0 = land

    Returns:
        scalar residual loss term (masked mean squared residual)
    """
    implied_tendency = (ohc_t_plus_dt - ohc_t) / dt_seconds
    flux_tendency = flux_t  # W/m^2 == J/(m^2 s), directly comparable to d(OHC)/dt
    residual = (implied_tendency - flux_tendency) ** 2
    masked_residual = residual * mask
    return masked_residual.sum() / mask.sum().clamp_min(1.0)

In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: lightweight ensemble / diffusion-flavoured stochastic head.
# Several independently-seeded stochastic forward passes per step, read as an
# ensemble spread -- a cheap probe for whether equilibrium variance drift is
# present, before committing to a full diffusion sampler.
# -----------------------------------------------------------------------------

class StochasticForwardUNet(nn.Module):
    """
    Noise-conditioned stochastic wrapper around ForwardUNet.

    Concatenates a fixed-channel noise map to the input at each forward call,
    so that repeated calls with different noise produce a distribution of
    plausible next states rather than a single deterministic prediction. This
    is a deliberately lightweight stand-in for a full diffusion sampler --
    appropriate for cheaply checking whether ensemble spread under repeated
    (control) forcing already explains observed drift, before investing in a
    full denoising-diffusion training procedure.
    """

    def __init__(self, n_prior_states: int = 2, n_forcing_channels: int = 1, n_noise_channels: int = 4):
        super().__init__()
        self.n_noise_channels = n_noise_channels
        self.base_unet = UNet(
            input_channel_count=n_prior_states + n_forcing_channels + n_noise_channels,
            output_channel_count=1,
        )

    def forward(self, prior_states: torch.Tensor, forcing: torch.Tensor, n_samples: int = 1) -> torch.Tensor:
        """
        Returns (n_samples, B, 1, H, W) -- an ensemble of predictions per input.
        """
        B, _, H, W = prior_states.shape
        samples = []
        for _ in range(n_samples):
            noise = torch.randn(B, self.n_noise_channels, H, W, device=prior_states.device)
            x = torch.cat([prior_states, forcing, noise], dim=1)
            samples.append(self.base_unet(x))
        return torch.stack(samples, dim=0)

In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: closed-loop drift-nudging term, RL-adjacent.
# A soft bias-correction term nudging the rollout's running mean toward a
# climatological constraint, rather than a full RL controller with its own
# reward formulation -- see the framing note above on why this is the lighter-
# weight option to try first.
# -----------------------------------------------------------------------------

def climatological_nudge_term(
    rollout_states: torch.Tensor,
    climatology: torch.Tensor,
    mask: torch.Tensor,
) -> torch.Tensor:
    """
    Penalises systematic drift of the rollout's running mean away from a
    precomputed climatological OHC field, applied as an additional term over
    a full rollout (not a per-step loss).

    Args:
        rollout_states : (N, B, 1, H, W) -- N-step rollout
        climatology     : (1, 1, H, W) -- precomputed long-run mean OHC field,
                           e.g. the training-period climatology already computed
                           as part of build_normalisation above
        mask            : (H, W) ocean mask

    Returns:
        scalar nudge loss term
    """
    rollout_mean = rollout_states.mean(dim=0)  # (B, 1, H, W)
    deviation = (rollout_mean - climatology) ** 2
    masked_deviation = deviation * mask
    return masked_deviation.sum() / mask.sum().clamp_min(1.0)

## Multi-step rollout loss

Single-step masked MSE, as used in `AutoEncoder_om2.ipynb`, is replaced with a
multi-step recurrent rollout loss, following Subel & Zanna (2024, Eq. 1) and Samudra
(Eq. 2). With a single-step loss the network never sees its own compounding error
during training, and for a slow field like OHC a single monthly step carries little
signal. Subel & Zanna found exactly this for temperature: short training windows
produced poor or unstable long-horizon skill, fixed by increasing either the timestep
or the number of recurrent rollout passes N.

At monthly cadence, N=4 is already a 4-month training horizon -- proportionally longer
than Subel & Zanna's daily-data N=4 (4 days) or Samudra's 5-day-step N=4 (20 days),
so this default is reasonable but left tunable: the right N should be cross-validated
against the OHC field's own decorrelation time, not copied from either paper.

In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: multi-step recurrent rollout loss with optional PINN / diffusion /
# RL-nudge extensions. Replaces the single-step masked MSE from AutoEncoder_om2.
# loss_mode switch wires the three frontier variants in without changing the
# training loop structure -- see the architecture section for each variant's
# motivating failure mode.
# -----------------------------------------------------------------------------

def rollout_loss(
    model,
    initial_prior_states,
    forcing_sequence,
    target_sequence,
    mask,
    n_steps=4,
    loss_mode="baseline",
    pinn_weight=0.05,
    nudge_weight=0.05,
    climatology=None,
    dt_seconds=2.628e6,
    n_ensemble=1,
):
    """
    Multi-step recurrent rollout loss.

    The model's own predictions are fed back as input for n_steps passes,
    accumulating loss over every step in the rollout window (not just the last),
    so that compounding errors are visible during training.

    The loss_mode switch adds optional auxiliary terms on top of the base MSE:
      "baseline"  -- MSE rollout loss only (deterministic ForwardUNet).
      "pinn"      -- adds heat_budget_residual weighted by pinn_weight.
      "diffusion" -- uses StochasticForwardUNet; MSE averaged over ensemble.
      "rl_nudge"  -- adds climatological_nudge_term weighted by nudge_weight.
    """
    prior_states = initial_prior_states
    mse_total = 0.0
    pinn_total = 0.0
    rollout_states = []

    for step in range(n_steps):
        forcing_t = forcing_sequence[:, step : step + 1]
        target_t  = target_sequence[:, step : step + 1]

        if loss_mode == "diffusion":
            ensemble_pred = model(prior_states, forcing_t, n_samples=n_ensemble)
            pred_t = ensemble_pred.mean(dim=0)
            per_member_err = (ensemble_pred - target_t.unsqueeze(0)) ** 2
            step_mse = (per_member_err * mask).sum() / mask.sum().clamp_min(1.0) / n_ensemble
        else:
            pred_t   = model(prior_states, forcing_t)
            step_err = (pred_t - target_t) ** 2
            step_mse = (step_err * mask).sum() / mask.sum().clamp_min(1.0)

        mse_total = mse_total + step_mse
        rollout_states.append(pred_t)

        if loss_mode == "pinn":
            pinn_total = pinn_total + heat_budget_residual(
                ohc_t=prior_states[:, 1:2],
                ohc_t_plus_dt=pred_t,
                flux_t=forcing_t,
                dt_seconds=dt_seconds,
                mask=mask,
            )

        # Feed prediction back in as context for the next step.
        prior_states = torch.cat([prior_states[:, 1:2], pred_t], dim=1)

    total_loss = mse_total

    if loss_mode == "pinn":
        total_loss = total_loss + pinn_weight * pinn_total

    if loss_mode == "rl_nudge":
        if climatology is None:
            raise ValueError("climatology required for loss_mode==\"rl_nudge\"")
        rollout_tensor = torch.stack(rollout_states, dim=0)
        total_loss = total_loss + nudge_weight * climatological_nudge_term(
            rollout_tensor, climatology, mask
        )

    return total_loss

## Lightning wrapper for autoregressive training

`LightningWrapper` from `Emulator.py` was built for single-frame reconstruction and is
not reused directly here. `ForwardLightningWrapper` below carries over the same
Lightning conventions (constructor signature, Adam optimiser, masked loss) and adapts
the training step to unpack the nested pipeline output and call `rollout_loss`.

In [ ]:
class ForwardLightningWrapper(L.LightningModule):
    """
    Lightning wrapper for the autoregressive forward emulator.

    Expects each batch as ((prior_states, posterior_states), forcing):
      prior_states     (B, 2, H, W)         -- OHC(t-dt), OHC(t)
      posterior_states (B, n_steps, H, W)   -- ground-truth targets
      forcing          (B, n_steps, H, W)   -- surface flux over rollout window
    """

    def __init__(
        self,
        model,
        mask,
        lr=1e-4,
        loss_mode="baseline",
        n_steps=4,
        pinn_weight=0.05,
        nudge_weight=0.05,
        climatology=None,
        dt_seconds=2.628e6,
        n_ensemble=1,
    ):
        super().__init__()
        self.model        = model
        self.register_buffer("mask", torch.as_tensor(mask, dtype=torch.float32))
        self.lr           = lr
        self.loss_mode    = loss_mode
        self.n_steps      = n_steps
        self.pinn_weight  = pinn_weight
        self.nudge_weight = nudge_weight
        self.dt_seconds   = dt_seconds
        self.n_ensemble   = n_ensemble
        if climatology is not None:
            self.register_buffer("climatology", torch.as_tensor(climatology, dtype=torch.float32))
        else:
            self.climatology = None

    def _step(self, batch):
        (prior_states, posterior_states), forcing = batch
        return rollout_loss(
            model=self.model,
            initial_prior_states=prior_states,
            forcing_sequence=forcing,
            target_sequence=posterior_states,
            mask=self.mask,
            n_steps=self.n_steps,
            loss_mode=self.loss_mode,
            pinn_weight=self.pinn_weight,
            nudge_weight=self.nudge_weight,
            climatology=self.climatology,
            dt_seconds=self.dt_seconds,
            n_ensemble=self.n_ensemble,
        )

    def training_step(self, batch, batch_idx):
        loss = self._step(batch)
        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss = self._step(batch)
        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def configure_optimizers(self):
        return optim.Adam(self.model.parameters(), lr=self.lr)

In [ ]:
# Switch between architecture variants here.
# "baseline"  -> deterministic ForwardUNet, MSE rollout only (run this first).
# "pinn"      -> ForwardUNet + heat-budget consistency term.
# "diffusion" -> StochasticForwardUNet + ensemble-averaged MSE.
# "rl_nudge"  -> ForwardUNet + climatological nudge term.
# Per the agreed plan, run "baseline" first, inspect drift diagnostics below,
# then select the variant whose failure-mode match motivates it.

loss_mode = "baseline"

base_model = (
    StochasticForwardUNet(n_prior_states=2, n_forcing_channels=1)
    if loss_mode == "diffusion"
    else ForwardUNet(n_prior_states=2, n_forcing_channels=1)
)

lightning_model = ForwardLightningWrapper(
    model=base_model,
    mask=mask.values,
    lr=1e-4,
    loss_mode=loss_mode,
    n_steps=4,
    climatology=(
        mean["ocean_heat_content_2d"].values if loss_mode == "rl_nudge" else None
    ),
)


## Data module and fast dataloaders

Same general pattern as `AutoEncoder_om2.ipynb`. `PipelineLightningDataModule` wraps
`pipeline_i`, and `make_fast_dl` loads the full dataloader into memory once to avoid
PET's slow per-epoch reload (same workaround, same GitHub issue as the original
notebook). The only structural difference is that each `pipeline_i[date]` call now
returns the nested `((prior_states, posterior_states), forcing)` tuple rather than a
single frame; both `PipelineLightningDataModule` and `make_fast_dl` operate on whatever
shape the pipeline emits per index, so neither call needs to change.

In [ ]:
datamodule = pyearthtools.training.data.lightning.PipelineLightningDataModule(
    pipeline_i,
    **splits,
    batch_size=32,
    num_workers=0,
)


In [ ]:
%%time
datamodule.setup("fit")

fast_train_dl = make_fast_dl(
    datamodule.train_dataloader(),
    batch_size=32,
    shuffle=False,
    drop_last=False,
)

fast_valid_dl = make_fast_dl(
    datamodule.val_dataloader(),
    batch_size=32,
    shuffle=False,
    drop_last=False,
)


## Trainer

Unchanged from `AutoEncoder_om2.ipynb` -- same `L.Trainer` configuration and same
reason for bypassing PET's native `pyearthtools.training.lightning.Train` (too slow;
GitHub issue linked in the original notebook).

In [ ]:
# PET's native training (pyearthtools.training.lightning.Train) is too slow
# for this setup -- see https://github.com/ACCESS-Community-Hub/PyEarthTools/issues/265

trainer = L.Trainer(
    max_epochs=200,
    num_sanity_val_steps=0,
    accelerator="gpu",
    devices=1,
    logger=False,
    enable_checkpointing=False,
    enable_model_summary=False,
)

trainer.fit(
    model=lightning_model,
    train_dataloaders=fast_train_dl,
    val_dataloaders=fast_valid_dl,
)


## Skill-test rollout: held-out real forcing

An autoregressive rollout against the held-out `skill_test_split` window, using real
surface flux forcing. This measures whether the emulator tracks genuine forced OHC
trends over the test period. The rollout loop below mirrors Samudra's "8-year rollout"
in structure: start from a single initial condition drawn from the test window, then
step forward autoregressively, supplying the real flux at each step but never injecting
any ground-truth OHC back in.

The key diagnostic from this rollout is **trend-magnitude attenuation**: how much does
the emulator's OHC trend over the test period differ (in %, signed) from the OM2
ground-truth trend? If the model is systematically damping the real signal, that points
toward the PINN heat-budget term as the best-motivated next variant to try.

In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: autoregressive skill-test rollout against real held-out forcing.
# Runs the trained model forward from a single initial condition, supplying
# real flux at each step. Stores predictions and ground-truth for diagnostics.
# -----------------------------------------------------------------------------

base_model.eval()
base_model.to(device)

# Build a skill-test pipeline over the held-out real-forcing window.
# Same structure as pipeline_i but with skill_test_split as the iterator.
skill_pipeline = petpipe.Pipeline(
    ACCESS_OHC_accessor,
    petpipe.branching.PipelineBranchPoint(
        (*state_steps, state_window),
        forcing_steps,
    ),
    iterator=skill_test_split,
)

skill_preds = []
skill_targets = []

with torch.no_grad():
    prior_states = None
    for date in skill_test_split:
        (prior_window, target_window), forcing_window = skill_pipeline[date]
        # On the first step, seed from ground truth.
        if prior_states is None:
            prior_states = torch.tensor(
                prior_window, dtype=torch.float32, device=device
            ).unsqueeze(0)

        forcing_t = torch.tensor(
            forcing_window, dtype=torch.float32, device=device
        ).unsqueeze(0)
        target_t  = torch.tensor(
            target_window, dtype=torch.float32, device=device
        ).unsqueeze(0)

        if loss_mode == "diffusion":
            ensemble = base_model(prior_states, forcing_t, n_samples=8)
            pred_t   = ensemble.mean(dim=0)
        else:
            pred_t = base_model(prior_states, forcing_t)

        skill_preds.append(pred_t.cpu().numpy())
        skill_targets.append(target_t.cpu().numpy())

        # Autoregressive: feed prediction back in, not ground truth.
        prior_states = torch.cat([prior_states[:, 1:2], pred_t], dim=1)

skill_preds   = np.concatenate(skill_preds,   axis=0)
skill_targets = np.concatenate(skill_targets, axis=0)
print(f'Skill-test rollout complete. Prediction array: {skill_preds.shape}')

## Control rollout: repeated flat forcing (isolates equilibrium drift)

An autoregressive rollout driven by the repeated, near-zero-net-flux control window
(`control_iterator`), rather than real forcing. This is Samudra's key evaluation
innovation: by running the model under forcing that carries minimal net heat input,
any long-term drift in the emulator's OHC trajectory must be a property of the model
itself (compounding numerical or architectural error), not a response to a real trend in
the forcing.

This rollout produces no "ground truth" to compare against in the usual sense -- the
diagnostic is the trajectory's own statistics: does the long-run mean OHC drift
systematically upward or downward, and does the variance inflate or collapse? These
directly distinguish the two failure modes discussed earlier:
- **Systematic mean drift** → compounding bias in the recurrence, possibly addressable
  by the climatological nudge term.
- **Variance inflation** → the stochastic ensemble spread growing unsustainably, the
  failure mode a diffusion-style model is specifically built to quantify honestly rather
  than mask with a damped deterministic mean.

In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: long control rollout under repeated near-zero-net-flux forcing,
# to isolate equilibrium drift from genuine forced trend response.
# control_iterator chains the same 10-year DateRange n_control_repeats times
# via SuperIterator -- see the splits section for the rationale.
# -----------------------------------------------------------------------------

# Build a control pipeline using the repeated flat-forcing iterator.
control_pipeline = petpipe.Pipeline(
    ACCESS_OHC_accessor,
    petpipe.branching.PipelineBranchPoint(
        (*state_steps, state_window),
        forcing_steps,
    ),
    iterator=control_iterator,
)

control_preds = []
# For the diffusion variant, also store per-member predictions for spread.
control_ensemble = []  # populated only when loss_mode == 'diffusion'

with torch.no_grad():
    prior_states = None
    for date in control_iterator:
        (prior_window, _), forcing_window = control_pipeline[date]
        if prior_states is None:
            prior_states = torch.tensor(
                prior_window, dtype=torch.float32, device=device
            ).unsqueeze(0)

        forcing_t = torch.tensor(
            forcing_window, dtype=torch.float32, device=device
        ).unsqueeze(0)

        if loss_mode == "diffusion":
            ensemble = base_model(prior_states, forcing_t, n_samples=8)  # (8, 1, 1, H, W)
            pred_t   = ensemble.mean(dim=0)
            control_ensemble.append(ensemble.cpu().numpy())
        else:
            pred_t = base_model(prior_states, forcing_t)

        control_preds.append(pred_t.cpu().numpy())
        prior_states = torch.cat([prior_states[:, 1:2], pred_t], dim=1)

control_preds = np.concatenate(control_preds, axis=0)
print(f'Control rollout complete. Array shape: {control_preds.shape}')
if control_ensemble:
    control_ensemble = np.concatenate(control_ensemble, axis=1)  # (n_members, T, 1, H, W)
    print(f'Ensemble shape: {control_ensemble.shape}')

## Drift diagnostics

Two complementary diagnostics, matching Samudra's evaluation framework:

1. **Trend-attenuation diagnostic** from the skill-test rollout: the linear trend of
   area-weighted OHC in the emulator vs. OM2 ground truth, and the attenuation ratio
   (emulator trend / ground-truth trend). Samudra reported 20–50% attenuation for most
   depths in their 8-year test window; measuring the equivalent here establishes
   whether that failure mode is present in the 2D depth-integrated case, at what
   magnitude, and whether it is spatially uniform or concentrated in specific regions.
2. **Equilibrium drift diagnostic** from the control rollout: the linear trend and
   variance of the globally integrated OHC under flat/repeated forcing. Under ideal
   conditions (a model with no compounding bias), both should be near zero. Persistent
   trend or growing variance here is a property of the emulator itself, not the forcing,
   and establishes the baseline against which any PINN / diffusion / RL-nudge variant
   must improve.

In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: compute and tabulate the two core drift diagnostics.
# Un-normalise both rollouts back to physical OHC units using the same
# normalisation parameters used during training.
# -----------------------------------------------------------------------------

# Un-normalise back to physical units (J/m^2).
# Normalisation was Spatial_climatology, so undo: x_phys = x_norm * deviation + mean
ohc_mean = mean['ocean_heat_content_2d'].values[np.newaxis, :, :]
ohc_std  = deviation['ocean_heat_content_2d'].values[np.newaxis, :, :]
mask_np  = mask.values  # (H, W), 1=ocean 0=land

# skill_preds/skill_targets both shaped (T, 1, H, W) -> squeeze to (T, H, W)
pred_phys   = skill_preds[:, 0]   * ohc_std + ohc_mean
target_phys = skill_targets[:, 0] * ohc_std + ohc_mean
ctrl_phys   = control_preds[:, 0] * ohc_std + ohc_mean

# Area-weighted global mean (mask acts as area weight proxy at 1-degree resolution)
ocean_cells = mask_np.sum()
pred_global   = (pred_phys   * mask_np).sum(axis=(1,2)) / ocean_cells
target_global = (target_phys * mask_np).sum(axis=(1,2)) / ocean_cells
ctrl_global   = (ctrl_phys   * mask_np).sum(axis=(1,2)) / ocean_cells

# Linear trend fits
T_skill = len(pred_global)
T_ctrl  = len(ctrl_global)
t_skill = np.arange(T_skill)
t_ctrl  = np.arange(T_ctrl)

def lin_trend(t, y):
    p = np.polyfit(t, y, 1)
    return p[0]  # slope per timestep (here: per month)

pred_trend   = lin_trend(t_skill, pred_global)
target_trend = lin_trend(t_skill, target_global)
ctrl_trend   = lin_trend(t_ctrl,  ctrl_global)
ctrl_std     = ctrl_global.std()

attenuation_ratio = pred_trend / target_trend if target_trend != 0 else float('nan')

print('--- Drift diagnostics ---')
print(f'Skill-test | OM2 trend:      {target_trend:.4e} J/m^2/month')
print(f'Skill-test | Emulator trend: {pred_trend:.4e} J/m^2/month')
print(f'Skill-test | Attenuation:    {attenuation_ratio:.3f} (1.0 = perfect, <1 = damped)')
print()
print(f'Control    | Drift trend:    {ctrl_trend:.4e} J/m^2/month (should be ~0)')
print(f'Control    | Std dev:        {ctrl_std:.4e} J/m^2 (should be stable)')

## Plots

Two plot panels mirroring the evaluation structure above.

In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: skill-test rollout plot -- global mean OHC trajectory,
# emulator vs OM2 ground truth, with linear trend lines.
# Style mirrors the evaluation plots in AutoEncoder_om2.ipynb (cell 17).
# -----------------------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Panel 1: global mean OHC time series
ax = axes[0]
ax.plot(target_global, label='OM2 ground truth', color='steelblue')
ax.plot(pred_global,   label=f'Emulator ({loss_mode})', color='tomato', linestyle='--')
ax.plot(np.polyval(np.polyfit(t_skill, target_global, 1), t_skill),
        color='steelblue', linewidth=0.7, linestyle=':')
ax.plot(np.polyval(np.polyfit(t_skill, pred_global, 1), t_skill),
        color='tomato',    linewidth=0.7, linestyle=':')
ax.set_title('Skill-test: global mean OHC')
ax.set_xlabel('Months into rollout')
ax.set_ylabel('OHC (J/m^2)')
ax.legend()
ax.text(0.02, 0.05,
        f'Attenuation: {attenuation_ratio:.3f}',
        transform=ax.transAxes, fontsize=9)

# Panel 2: snapshot map of final-step error
ax = axes[1]
final_err = (pred_phys[-1] - target_phys[-1]) * mask_np
vmax = np.abs(final_err).max()
im = ax.imshow(final_err, cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='lower')
plt.colorbar(im, ax=ax, label='J/m^2')
ax.set_title(f'Final-step error (month {T_skill})')

plt.suptitle(f'Skill-test rollout  |  loss_mode={loss_mode}  |  N_steps={lightning_model.n_steps}',
             fontsize=11)
plt.tight_layout()
plt.savefig('skill_test_rollout.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: control rollout plot -- long-term global mean OHC trajectory
# under flat/repeated forcing, plus ensemble spread if loss_mode=='diffusion'.
# -----------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(ctrl_global, color='darkslategray', linewidth=0.8,
        label='Emulator mean (control)')
ax.axhline(ctrl_global[0], color='gray', linewidth=0.6, linestyle='--',
           label='Initial OHC')

# Shade ensemble spread if available
if control_ensemble:
    # control_ensemble: (n_members, T, 1, H, W)
    ens_phys = control_ensemble[:, :, 0] * ohc_std[np.newaxis] + ohc_mean[np.newaxis]
    ens_global = (ens_phys * mask_np[np.newaxis]).sum(axis=(-2,-1)) / ocean_cells
    ens_lo = ens_global.min(axis=0)
    ens_hi = ens_global.max(axis=0)
    ax.fill_between(np.arange(T_ctrl), ens_lo, ens_hi,
                    alpha=0.25, color='darkslategray', label='Ensemble spread')

# Mark repeat boundaries
repeat_len = T_ctrl // n_control_repeats
for k in range(1, n_control_repeats):
    ax.axvline(k * repeat_len, color='goldenrod', linewidth=0.5, linestyle=':')

ax.set_title('Control rollout: equilibrium drift under flat forcing')
ax.set_xlabel('Months into control run')
ax.set_ylabel('OHC (J/m^2)')
ax.legend()
ax.text(0.02, 0.92,
        f'Drift trend: {ctrl_trend:.3e} J/m^2/month\nStd dev: {ctrl_std:.3e} J/m^2',
        transform=ax.transAxes, fontsize=9, verticalalignment='top')

plt.suptitle(f'Control rollout  |  loss_mode={loss_mode}  |'
             f'  {n_control_repeats} x {repeat_len}-month cycles',
             fontsize=11)
plt.tight_layout()
plt.savefig('control_rollout.png', dpi=150, bbox_inches='tight')
plt.show()

## Closing note: reading the diagnostics and choosing the next variant

The two diagnostic plots and the printed attenuation ratio / drift trend are the
decision inputs for which `loss_mode` variant to try next.

**If attenuation ratio < 1 and control drift is near zero:**
The model is stable but systematically damping the forced trend -- this is the
trend-attenuation failure mode. Switch `loss_mode = "pinn"` and tune `pinn_weight`
(start at 0.05, increase if the attenuation ratio doesn't improve). The heat-budget
consistency term directly penalises the discrepancy between implied d(OHC)/dt and
the supplied flux, giving the model an incentive to match cumulative heat input
rather than regressing to the mean.

**If control drift trend is large or variance inflates over the control run:**
The model is compounding errors across the autoregressive chain -- this is the
equilibrium drift failure mode. Switch `loss_mode = "diffusion"` to expose this
honestly via ensemble spread, and inspect whether the ensemble mean stays closer
to the initial condition than the deterministic trajectory does. If the spread
tracks the drift well, that is evidence that a full diffusion sampler (trained with
denoising score matching, as in GenCast-style models) is the right next step.

**If both failure modes are present:**
Try `loss_mode = "rl_nudge"` as a cheap baseline for the equilibrium case --
the climatological nudge term adds a penalty when the rollout's running mean
strays from the training-period climatology, which is the lightest-weight
closed-loop correction available before investing in a full RL reward formulation.
Tune `nudge_weight` conservatively (0.01--0.1); too large and it will dominate
the MSE term and force the model to suppress any real trend as well as drift.

In all cases: run each variant for the same number of epochs as the baseline, compare
attenuation ratios and control-drift trends side by side, and prefer the variant
that improves *both* diagnostics simultaneously -- a variant that fixes trend
attenuation by increasing control drift has simply traded one failure mode for another.
